# PayShield AI — Real-Time Risk Prediction

## AI-Powered Payment Success Optimization

### Objective

Simulate real-time payment-system monitoring.

For a new bank monitoring window:

1. Receive monitoring features
2. Prepare the features
3. Generate risk probability
4. Classify risk level
5. Generate an alert
6. Explain the detected risk

This notebook simulates how PayShield could operate
when new payment-system data arrives.

In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv(
    "../data/processed/ml_dataset.csv"
)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (50944, 22)


,transaction_count,failure_rate,timeout_rate,avg_latency,max_latency,p95_latency,bank_error_rate,avg_amount,max_amount,hour,...,previous_failure_rate,previous_latency,previous_timeout_rate,failure_rate_change,latency_change,timeout_rate_change,rolling_failure_rate,rolling_latency,rolling_timeout_rate,risk_target
0,2,0.0,0.0,881.935,883.95,883.7485,0.0,244.72,320.59,0,...,0.0,1447.700,0.0,0.0,-565.765,0.0,0.0,1164.817500,0.0,0
1,1,0.0,0.0,882.880,882.88,882.8800,0.0,321.00,321.00,1,...,0.0,881.935,0.0,0.0,0.945,0.0,0.0,1070.838333,0.0,0
2,1,0.0,0.0,751.670,751.67,751.6700,0.0,395.70,395.70,1,...,0.0,882.880,0.0,0.0,-131.210,0.0,0.0,838.828333,0.0,0
3,1,0.0,0.0,1161.920,1161.92,1161.9200,0.0,32.14,32.14,1,...,0.0,751.670,0.0,0.0,410.250,0.0,0.0,932.156667,0.0,0
4,1,0.0,0.0,709.700,709.70,709.7000,0.0,920.47,920.47,2,...,0.0,1161.920,0.0,0.0,-452.220,0.0,0.0,874.430000,0.0,0


In [3]:
X = df.drop(columns=["risk_target"])
y = df["risk_target"]

print("Number of features:", X.shape[1])
print("Features:")
print(X.columns.tolist())

Number of features: 21
Features:
['transaction_count', 'failure_rate', 'timeout_rate', 'avg_latency', 'max_latency', 'p95_latency', 'bank_error_rate', 'avg_amount', 'max_amount', 'hour', 'day_of_week', 'is_weekend', 'previous_failure_rate', 'previous_latency', 'previous_timeout_rate', 'failure_rate_change', 'latency_change', 'timeout_rate_change', 'rolling_failure_rate', 'rolling_latency', 'rolling_timeout_rate']


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

print("Production model trained successfully!")

Production model trained successfully!


In [5]:
ALERT_THRESHOLD = 0.20

print(
    "PayShield alert threshold:",
    ALERT_THRESHOLD
)

PayShield alert threshold: 0.2


In [6]:
def get_risk_level(probability):

    if probability >= 0.80:
        return "HIGH"

    elif probability >= 0.50:
        return "MEDIUM"

    else:
        return "LOW"

In [7]:
def get_alert(probability):

    if probability >= 0.80:
        return "HIGH RISK ALERT"

    elif probability >= 0.50:
        return "WARNING"

    elif probability >= ALERT_THRESHOLD:
        return "MONITOR"

    else:
        return "NO ALERT"

In [8]:
new_window = X_test.iloc[[0]].copy()

new_window

,transaction_count,failure_rate,timeout_rate,avg_latency,max_latency,p95_latency,bank_error_rate,avg_amount,max_amount,hour,...,is_weekend,previous_failure_rate,previous_latency,previous_timeout_rate,failure_rate_change,latency_change,timeout_rate_change,rolling_failure_rate,rolling_latency,rolling_timeout_rate
35855,4,0.0,0.0,1064.24,1861.51,1748.2105,0.0,204.7275,309.03,20,...,0,0.0,1908.44,0.0,0.0,-844.2,0.0,0.0,1326.718333,0.0


In [9]:
risk_probability = rf_model.predict_proba(
    new_window
)[0, 1]

print(
    "Predicted risk probability:",
    round(risk_probability * 100, 2),
    "%"
)

Predicted risk probability: 0.0 %


In [10]:
risk = get_risk_level(
    risk_probability
)

alert = get_alert(
    risk_probability
)

print("Risk Level:", risk)
print("Alert:", alert)

Risk Level: LOW
Alert: NO ALERT


In [11]:
def predict_payment_risk(
    model,
    monitoring_window
):

    probability = model.predict_proba(
        monitoring_window
    )[0, 1]

    level = get_risk_level(
        probability
    )

    alert = get_alert(
        probability
    )

    return {
        "risk_probability": probability,
        "risk_level": level,
        "alert": alert
    }

In [12]:
prediction = predict_payment_risk(
    rf_model,
    new_window
)

prediction

{'risk_probability': np.float64(0.0), 'risk_level': 'LOW', 'alert': 'NO ALERT'}

In [13]:
def display_prediction(
    prediction
):

    probability = prediction[
        "risk_probability"
    ]

    level = prediction[
        "risk_level"
    ]

    alert = prediction[
        "alert"
    ]

    print("======================================")
    print("           PAYSHIELD AI")
    print("======================================")

    print(
        f"Risk Probability : {probability:.1%}"
    )

    print(
        f"Risk Level       : {level}"
    )

    print(
        f"System Status    : {alert}"
    )

    print("======================================")

In [14]:
display_prediction(
    prediction
)

           PAYSHIELD AI
Risk Probability : 0.0%
Risk Level       : LOW
System Status    : NO ALERT


In [15]:
sample_windows = X_test.sample(
    n=10,
    random_state=42
).copy()

sample_predictions = []

In [16]:
for index, row in sample_windows.iterrows():

    window = row.to_frame().T

    probability = rf_model.predict_proba(
        window
    )[0, 1]

    sample_predictions.append({
        "index": index,
        "risk_probability": probability,
        "risk_level": get_risk_level(probability),
        "alert": get_alert(probability)
    })

realtime_results = pd.DataFrame(
    sample_predictions
)

realtime_results

,index,risk_probability,risk_level,alert
0,15367,0.00,LOW,NO ALERT
1,21954,0.00,LOW,NO ALERT
2,37558,0.00,LOW,NO ALERT
3,47170,0.00,LOW,NO ALERT
4,27526,0.00,LOW,NO ALERT
5,47936,0.01,LOW,NO ALERT
6,34583,0.00,LOW,NO ALERT
7,35480,0.00,LOW,NO ALERT
8,46340,0.00,LOW,NO ALERT
9,3171,0.00,LOW,NO ALERT


In [17]:
realtime_results = (
    realtime_results
    .sort_values(
        "risk_probability",
        ascending=False
    )
    .reset_index(drop=True)
)

realtime_results

,index,risk_probability,risk_level,alert
0,47936,0.01,LOW,NO ALERT
1,15367,0.00,LOW,NO ALERT
2,21954,0.00,LOW,NO ALERT
3,37558,0.00,LOW,NO ALERT
4,47170,0.00,LOW,NO ALERT
5,27526,0.00,LOW,NO ALERT
6,34583,0.00,LOW,NO ALERT
7,35480,0.00,LOW,NO ALERT
8,46340,0.00,LOW,NO ALERT
9,3171,0.00,LOW,NO ALERT


In [18]:
def realtime_monitor(
    model,
    monitoring_data
):

    output = []

    for index, row in monitoring_data.iterrows():

        window = row.to_frame().T

        probability = model.predict_proba(
            window
        )[0, 1]

        output.append({
            "index": index,
            "risk_probability": probability,
            "risk_level": get_risk_level(
                probability
            ),
            "alert": get_alert(
                probability
            )
        })

    return (
        pd.DataFrame(output)
        .sort_values(
            "risk_probability",
            ascending=False
        )
        .reset_index(drop=True)
    )

In [22]:
# Generate probabilities for the complete test set

test_probabilities = rf_model.predict_proba(
    X_test
)[:, 1]

# Create monitoring results
all_results = X_test.copy()

all_results["risk_probability"] = test_probabilities

all_results["risk_level"] = (
    all_results["risk_probability"]
    .apply(get_risk_level)
)

all_results["alert"] = (
    all_results["risk_probability"]
    .apply(get_alert)
)

# Show the 20 highest-risk windows
live_results = (
    all_results
    .sort_values(
        "risk_probability",
        ascending=False
    )
    .head(20)
    .reset_index()
)

live_results[
    [
        "index",
        "risk_probability",
        "risk_level",
        "alert"
    ]
]

,index,risk_probability,risk_level,alert
0,23698,0.990,HIGH,HIGH RISK ALERT
1,39835,0.985,HIGH,HIGH RISK ALERT
2,39846,0.985,HIGH,HIGH RISK ALERT
3,2433,0.980,HIGH,HIGH RISK ALERT
4,46656,0.980,HIGH,HIGH RISK ALERT
5,16922,0.980,HIGH,HIGH RISK ALERT
6,2414,0.975,HIGH,HIGH RISK ALERT
7,23681,0.975,HIGH,HIGH RISK ALERT
8,39854,0.975,HIGH,HIGH RISK ALERT
9,39862,0.970,HIGH,HIGH RISK ALERT


In [27]:
def get_risk_level(probability):

    if probability >= 0.80:
        return "HIGH"

    elif probability >= 0.50:
        return "MEDIUM"

    elif probability >= 0.20:
        return "ELEVATED"

    else:
        return "LOW"

In [28]:
def get_alert(probability):

    if probability >= 0.80:
        return "HIGH RISK ALERT"

    elif probability >= 0.50:
        return "WARNING"

    elif probability >= 0.20:
        return "MONITOR"

    else:
        return "NO ALERT"

In [30]:
# ============================================
# PAYSHIELD REAL-TIME DEMO
# ============================================

# 1. Generate probabilities for all test data
test_probabilities = rf_model.predict_proba(X_test)[:, 1]

# 2. Create a fresh results dataframe
demo = X_test.copy()

demo["risk_probability"] = test_probabilities

# 3. Create risk levels directly
demo["risk_level"] = "LOW"

demo.loc[
    demo["risk_probability"] >= 0.20,
    "risk_level"
] = "ELEVATED"

demo.loc[
    demo["risk_probability"] >= 0.50,
    "risk_level"
] = "MEDIUM"

demo.loc[
    demo["risk_probability"] >= 0.80,
    "risk_level"
] = "HIGH"

# 4. Create alerts directly
demo["alert"] = "NO ALERT"

demo.loc[
    demo["risk_probability"] >= 0.20,
    "alert"
] = "MONITOR"

demo.loc[
    demo["risk_probability"] >= 0.50,
    "alert"
] = "WARNING"

demo.loc[
    demo["risk_probability"] >= 0.80,
    "alert"
] = "HIGH RISK ALERT"

# 5. Create three groups
low = demo[
    demo["risk_probability"] < 0.20
]

medium = demo[
    (demo["risk_probability"] >= 0.20) &
    (demo["risk_probability"] < 0.80)
]

high = demo[
    demo["risk_probability"] >= 0.80
]

# 6. Take samples safely
parts = []

if len(low) > 0:
    parts.append(
        low.sample(
            min(5, len(low)),
            random_state=42
        )
    )

if len(medium) > 0:
    parts.append(
        medium.sample(
            min(5, len(medium)),
            random_state=42
        )
    )

if len(high) > 0:
    parts.append(
        high.sample(
            min(5, len(high)),
            random_state=42
        )
    )

# 7. Combine everything
if len(parts) > 0:
    demo_results = pd.concat(
        parts,
        ignore_index=True
    )
else:
    demo_results = pd.DataFrame()

# 8. Sort
if len(demo_results) > 0:
    demo_results = demo_results.sort_values(
        "risk_probability",
        ascending=False
    )

# 9. Display
print("LOW:", len(low))
print("ELEVATED + MEDIUM:", len(medium))
print("HIGH:", len(high))

print("\nPayShield Demo Results:")

demo_results[
    [
        "risk_probability",
        "risk_level",
        "alert"
    ]
]

LOW: 10138
ELEVATED + MEDIUM: 11
HIGH: 40

PayShield Demo Results:


,risk_probability,risk_level,alert
13,0.975,HIGH,HIGH RISK ALERT
12,0.960,HIGH,HIGH RISK ALERT
14,0.955,HIGH,HIGH RISK ALERT
10,0.855,HIGH,HIGH RISK ALERT
11,0.800,HIGH,HIGH RISK ALERT
8,0.650,MEDIUM,WARNING
6,0.370,ELEVATED,MONITOR
5,0.315,ELEVATED,MONITOR
9,0.315,ELEVATED,MONITOR
7,0.205,ELEVATED,MONITOR


In [23]:
print(
    live_results["alert"]
    .value_counts()
)

alert
HIGH RISK ALERT    20
Name: count, dtype: int64


In [24]:
display_columns = [
    "index",
    "risk_probability",
    "risk_level",
    "alert"
]

display_df = live_results[display_columns].copy()

display_df["risk_probability"] = (
    display_df["risk_probability"] * 100
).round(2)

display_df

,index,risk_probability,risk_level,alert
0,23698,99.0,HIGH,HIGH RISK ALERT
1,39835,98.5,HIGH,HIGH RISK ALERT
2,39846,98.5,HIGH,HIGH RISK ALERT
3,2433,98.0,HIGH,HIGH RISK ALERT
4,46656,98.0,HIGH,HIGH RISK ALERT
5,16922,98.0,HIGH,HIGH RISK ALERT
6,2414,97.5,HIGH,HIGH RISK ALERT
7,23681,97.5,HIGH,HIGH RISK ALERT
8,39854,97.5,HIGH,HIGH RISK ALERT
9,39862,97.0,HIGH,HIGH RISK ALERT


In [21]:
live_results.to_csv(
    "../data/processed/realtime_predictions.csv",
    index=False
)

print(
    "Real-time prediction results saved successfully!"
)

Real-time prediction results saved successfully!
